# 3. Testing the similarity median correction

This notebook tests one question: does median-based calibration remove useful similarity evidence from potentially matching feature titles? The test matters because a below-median score is clipped to zero by the current method even though it may still contain useful evidence. The only tested change is whether median correction is applied; no alternative weights or fixed cutoffs are tested.

A **title pair** means two different feature-title labels selected from the 445 unique titles—for example, one row might compare “Waterproof” with “Windproof.” It is not a pair of products, and it is not a title matched with one of its descriptions. There are 445 × 444 / 2 = 98,790 possible unordered pairs of distinct titles. For each pair, the notebook calculates how similar the two titles appear under the semantic and fuzzy signals, then checks whether those scores fall below the corresponding medians.

**Current method:** For semantic cosine and fuzzy title scores separately, calculate the median across all 98,790 distinct title pairs, subtract that median, and divide by `1 - median` to scale the remaining values to 0–1. Scores at or below the median become zero. **Control:** do not use the observed median: map semantic cosine from its theoretical range [-1, 1] to [0, 1], and leave fuzzy scores on their original [0, 1] scale. The notebook reproduces the same embedding, TF-IDF, fuzzy-title, and winter/wind penalty steps as `3_similarity.ipynb`; original 50/25/25 blend weights and all other settings stay unchanged.

Semantic similarity uses title and aligned-description embeddings, so it can recognize related wording when titles differ. Fuzzy similarity compares title strings only, so it responds more strongly to literal wording overlap. The review sample is stratified by whether each raw signal falls above or at/below its median, allowing us to see when the two signals disagree.

The sample contains up to six pairs from each of the four semantic/fuzzy median groups (at most 24). The pairs were judged on whether the titles name the same underlying property. This is a small pilot, not a statistically conclusive evaluation. Both matrices are kept in memory; this notebook does **not** overwrite `data/similarity.npz` or rerun clustering.

In [1]:
import json
import re
from collections import Counter
from pathlib import Path
from pprint import pprint

import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import display
from rapidfuzz import fuzz, process, utils
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

# These are copied unchanged from 3_similarity.ipynb.
W_SEMANTIC, W_TFIDF, W_FUZZY = 0.50, 0.25, 0.25
W_TITLE = 0.50
PENALTY = 0.30
MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
EMBEDDING_PROMPT = "Instruct: Identify product features that describe the same property\nQuery: "

## 1. Load the same feature records

The test uses the same 445 feature titles, aligned descriptions, and unpaired descriptions as the production similarity notebook. No titles or descriptions are filtered for this comparison.

In [2]:
feature_data = json.loads(Path("data/features_en.json").read_text(encoding="utf8"))
attribute_records = feature_data["attribute_records"]
unpaired_descriptions = feature_data["unpaired_descriptions"]

attributes = [record["title"] for record in attribute_records]
descriptions = [record["descriptions"] for record in attribute_records]

print(
    f"{len(attributes)} titles, "
    f"{sum(bool(texts) for texts in descriptions)} with descriptions, "
    f"{len(unpaired_descriptions)} unpaired descriptions"
)

445 titles, 433 with descriptions, 22 unpaired descriptions


## 2. Reproduce TF-IDF exactly

TF-IDF is not median-calibrated in the original pipeline. It remains unchanged in both versions, so any difference between the two blended matrices comes from the semantic and fuzzy calibration being tested.

In [3]:
documents = [" ".join(texts) for texts in descriptions]

tfidf = TfidfVectorizer(stop_words="english", sublinear_tf=True)
tfidf.fit([document for document in documents if document] + unpaired_descriptions)
vectors = tfidf.transform(documents)
tfidf_similarity = (vectors @ vectors.T).toarray()

print("TF-IDF similarity matrix:", tfidf_similarity.shape)

TF-IDF similarity matrix: (445, 445)


## 3. Reproduce semantic vectors and compare median correction

The title and description embeddings are built exactly as in `3_similarity.ipynb`: same Qwen model and prompt, same description-occurrence weighting, same 50/50 title/context blend, and the same vector normalization. The model may need to download the first time this cell runs.

The current score uses the observed median as zero and clips every lower score to zero. The no-median control uses `(cosine + 1) / 2`, a fixed-range conversion that preserves the ordering and differences of all cosine scores without using the observed median.

In [4]:
embedding_model = SentenceTransformer(MODEL_NAME, model_kwargs={"dtype": torch.float32})
title_vectors = embedding_model.encode(
    attributes,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)

all_descriptions = sorted({text for texts in descriptions for text in texts})
description_embedding_vectors = embedding_model.encode(
    all_descriptions,
    prompt=EMBEDDING_PROMPT,
    normalize_embeddings=True,
    show_progress_bar=True,
)
description_vector_by_text = {
    text: description_embedding_vectors[index]
    for index, text in enumerate(all_descriptions)
}

attribute_vectors = []
for title_vector, texts in zip(title_vectors, descriptions):
    if texts:
        description_counts = Counter(texts)
        weighted_description_vector = np.average(
            [description_vector_by_text[text] for text in description_counts],
            axis=0,
            weights=list(description_counts.values()),
        )
        vector = W_TITLE * title_vector + (1 - W_TITLE) * weighted_description_vector
        vector = vector / np.linalg.norm(vector)
    else:
        vector = title_vector
    attribute_vectors.append(vector)

attribute_vectors = np.asarray(attribute_vectors)
raw_semantic = attribute_vectors @ attribute_vectors.T
upper_indices = np.triu_indices(len(attributes), k=1)
semantic_baseline = np.median(raw_semantic[upper_indices])
if semantic_baseline >= 1:
    raise ValueError("Semantic similarities cannot be calibrated: median baseline is 1.")

semantic_current = np.clip(
    (raw_semantic - semantic_baseline) / (1 - semantic_baseline),
    0,
    1,
)
semantic_no_median = np.clip((raw_semantic + 1) / 2, 0, 1)
print(f"Raw semantic median: {semantic_baseline:.3f}")
print(f"Distinct title pairs: {len(upper_indices[0]):,}")
print(f"Semantic pairs zeroed by median correction: {np.mean(semantic_current[upper_indices] == 0):.1%}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Raw semantic median: 0.566
Distinct title pairs: 98,790
Semantic pairs zeroed by median correction: 50.0%


## 4. Reproduce fuzzy title scores and compare median correction

Fuzzy title matching uses the same token-sort scorer and text processor as the original. The current version median-calibrates and clips it; the no-median control keeps its original score, which is already in [0, 1].

In [5]:
fuzzy_raw = process.cdist(
    attributes,
    attributes,
    scorer=fuzz.token_sort_ratio,
    processor=utils.default_process,
) / 100

fuzzy_baseline = np.median(fuzzy_raw[upper_indices])
fuzzy_current = np.clip(
    (fuzzy_raw - fuzzy_baseline) / (1 - fuzzy_baseline),
    0,
    1,
)
fuzzy_no_median = fuzzy_raw.copy()
print(f"Raw fuzzy median: {fuzzy_baseline:.3f}")
print(f"Fuzzy pairs zeroed by median correction: {np.mean(fuzzy_current[upper_indices] == 0):.1%}")

Raw fuzzy median: 0.318


Fuzzy pairs zeroed by median correction: 50.9%


## 5. Blend the current method and no-median control

Both versions use the same component matrices, original 50/25/25 weights, unchanged TF-IDF, and the same title-based winter-versus-windproof penalty. The only difference is whether semantic and fuzzy scores use median calibration. These are comparison matrices only: neither is saved or used to rerun clustering.

In [6]:
assert np.isclose(W_SEMANTIC + W_TFIDF + W_FUZZY, 1), "weights must sum to 1"

similarity_current = (
    W_SEMANTIC * semantic_current
    + W_TFIDF * tfidf_similarity
    + W_FUZZY * fuzzy_current
)
similarity_no_median = (
    W_SEMANTIC * semantic_no_median
    + W_TFIDF * tfidf_similarity
    + W_FUZZY * fuzzy_no_median
)

CONCEPTS = {
    "winter": r"\b(winter|cold|freez\w*|frost\w*)\b",
    "windproof": r"\b(wind\w*)\b",
}
CONFLICTS = [("winter", "windproof")]

concepts_by_attribute = [
    {
        name
        for name, pattern in CONCEPTS.items()
        if re.search(pattern, title, re.IGNORECASE)
    }
    for title in attributes
]

penalty_matrix = np.zeros_like(similarity_current)
for first_concept, second_concept in CONFLICTS:
    first_only = np.array(
        [first_concept in concepts and second_concept not in concepts for concepts in concepts_by_attribute]
    )
    second_only = np.array(
        [second_concept in concepts and first_concept not in concepts for concepts in concepts_by_attribute]
    )
    penalty_matrix[np.ix_(first_only, second_only)] = PENALTY
    penalty_matrix[np.ix_(second_only, first_only)] = PENALTY

similarity_current = np.clip(similarity_current - penalty_matrix, 0, 1)
similarity_no_median = np.clip(similarity_no_median - penalty_matrix, 0, 1)
for candidate in (similarity_current, similarity_no_median):
    np.fill_diagonal(candidate, 1.0)

print("Conflict-penalized pairs:", int(np.count_nonzero(penalty_matrix) // 2))

Conflict-penalized pairs: 72


## 6. Select a small, balanced review sample

The code first considers every possible unordered pair of distinct titles. For each pair it has two raw scores: a semantic score based on the title-and-description embeddings, and a fuzzy score based on how similar the title wording is. It compares each score with that signal's median across all 98,790 possible title pairs.

It then sorts the possible pairs into four groups: (1) both scores are at or below their medians, (2) only the semantic score is at or below its median, (3) only the fuzzy score is at or below its median, and (4) both scores are above their medians. These groups reveal which signal or signals the current correction would set to zero for a pair. The notebook randomly selects up to six pairs from each group—at most 24 total—with a fixed seed so the sample is reproducible.

Each sampled row shows the two feature-title labels and some of the descriptions collected for each title. Descriptions provide context when titles are short or ambiguous; they are not treated as extra title-pair items. The displayed description text is shortened for readability.

Because the groups are deliberately balanced rather than sampled in proportion to the full dataset, their combined label percentage is **not** an estimate of how common same-property pairs are overall. The purpose is to inspect examples on both sides of the cutoffs.

In [7]:
pair_i, pair_j = upper_indices
pair_groups = {}
for i, j in zip(pair_i, pair_j):
    semantic_zeroed = raw_semantic[i, j] <= semantic_baseline
    fuzzy_zeroed = fuzzy_raw[i, j] <= fuzzy_baseline
    semantic_side = "semantic_at_or_below_median" if semantic_zeroed else "semantic_above_median"
    fuzzy_side = "fuzzy_at_or_below_median" if fuzzy_zeroed else "fuzzy_above_median"
    stratum = f"{semantic_side} + {fuzzy_side}"
    pair_groups.setdefault(stratum, []).append({
        "i": int(i),
        "j": int(j),
        "stratum": stratum,
        "semantic_raw": float(raw_semantic[i, j]),
        "fuzzy_raw": float(fuzzy_raw[i, j]),
        "semantic_zeroed": bool(semantic_zeroed),
        "fuzzy_zeroed": bool(fuzzy_zeroed),
    })

rng = np.random.default_rng(42)
sampled_pairs = []
for stratum, group in sorted(pair_groups.items()):
    count = min(6, len(group))
    selected_positions = rng.choice(len(group), size=count, replace=False)
    sampled_pairs.extend(group[int(position)] for position in selected_positions)
sampled_pairs.sort(key=lambda pair: (pair["stratum"], pair["i"], pair["j"]))

def description_context(index):
    unique_texts = list(dict.fromkeys(descriptions[index]))
    context = " | ".join(unique_texts) if unique_texts else "(no aligned description)"
    return context[:220] + ("..." if len(context) > 220 else "")

review_rows = []
for pair_id, pair in enumerate(sampled_pairs):
    i, j = pair["i"], pair["j"]
    review_rows.append({
        "pair_id": pair_id,
        "stratum": pair["stratum"],
        "title_a": attributes[i],
        "title_b": attributes[j],
        "descriptions_a": description_context(i),
        "descriptions_b": description_context(j),
        "semantic_raw": pair["semantic_raw"],
        "semantic_current": float(semantic_current[i, j]),
        "semantic_no_median": float(semantic_no_median[i, j]),
        "fuzzy_raw": pair["fuzzy_raw"],
        "fuzzy_current": float(fuzzy_current[i, j]),
        "tfidf_unchanged": float(tfidf_similarity[i, j]),
        "final_current": float(similarity_current[i, j]),
        "final_no_median": float(similarity_no_median[i, j]),
        "semantic_zeroed": pair["semantic_zeroed"],
        "fuzzy_zeroed": pair["fuzzy_zeroed"],
    })

print("Pairs selected by group:")
pprint({stratum: min(6, len(group)) for stratum, group in sorted(pair_groups.items())})
print("Review the titles and descriptions, then compare the scores:")
pprint([
    {key: row[key] for key in ("pair_id", "stratum", "title_a", "title_b", "descriptions_a", "descriptions_b")}
    for row in review_rows
], width=140)
pprint([
    {key: (round(value, 3) if isinstance(value, float) else value) for key, value in row.items()
     if key not in ("title_a", "title_b", "descriptions_a", "descriptions_b")}
    for row in review_rows
], width=140)

Pairs selected by group:
{'semantic_above_median + fuzzy_above_median': 6,
 'semantic_above_median + fuzzy_at_or_below_median': 6,
 'semantic_at_or_below_median + fuzzy_above_median': 6,
 'semantic_at_or_below_median + fuzzy_at_or_below_median': 6}
Review the titles and descriptions, then compare the scores:
[{'descriptions_a': 'Enhancing performance by allowing for efficient movement',
  'descriptions_b': 'At only 18 grams, it adds virtually no bulk',
  'pair_id': 0,
  'stratum': 'semantic_above_median + fuzzy_above_median',
  'title_a': 'High stretch and anatomically optimised fit',
  'title_b': 'Ultra-lightweight design'},
 {'descriptions_a': 'Blending flexibility, breathability, and comfort',
  'descriptions_b': 'Seamlessly integrates under shorts, creating a casual yet functional cycling outfit',
  'pair_id': 1,
  'stratum': 'semantic_above_median + fuzzy_above_median',
  'title_a': 'Premium high-stretch material back',
  'title_b': 'Invisible under a loose short'},
 {'description

## 7. Record a quick human judgment

For each row, read `title_a` and `title_b` as the two things being compared, using their descriptions as context. Decide whether the titles name the **same underlying product property**—not simply whether they are both useful features or could appear on the same product. Mark `same` for synonyms or equivalent labels; mark `not_same` for different properties. For example (illustrative), “windproof” and “waterproof” are both protective features but describe different properties, so they would be `not_same`. If a title is too vague to judge even with its descriptions, leave that pair out.

Enter your judgments in the dictionary below using the displayed `pair_id` values. Leave a pair out if you cannot judge it. The next cell reports how often median correction zeroes each signal for pairs you marked `same` and `not_same`, compares the final scores with and without that correction, and plots the raw semantic and fuzzy scores with their medians.

The sample is small and intentionally balanced, so use it to identify obvious problems and decide whether a larger labeled test is worthwhile—not to claim a precise error rate.

In [8]:
# Replace None with "same" or "not_same" for the pairs you reviewed.
#DECIDE wether the titles describe the same underlaying product property:
review_labels = {
    0: "not_same",
    1: "not_same",
    2: "not_same",
    3: "not_same",
    4: "not_same",
    5: "not_same",
    6: "same", # property secure hold
    7: "not_same",
    8: "not_same",
    9: "same", #property lightweight shoes
    10: "not_same",
    11: "same", #property waterproofiness
    12: "not_same",
    13: "not_same",
    14: "not_same",
    15: "not_same",
    16: "not_same",
    17: "not_same",
    18: "not_same",
    19: "not_same",
    20: "not_same",
    21: "not_same",
    22: "not_same",
    23: "not_same"
}

invalid_labels = set(review_labels.values()) - {"same", "not_same"}
if invalid_labels:
    raise ValueError(f"Use only 'same' or 'not_same'; invalid labels: {invalid_labels}")

labeled = [row for row in review_rows if row["pair_id"] in review_labels]
if not labeled:
    print("No judgments entered yet. Fill in review_labels above and rerun this cell.")
else:
    same_pairs = [row for row in labeled if review_labels[row["pair_id"]] == "same"]
    not_same_pairs = [row for row in labeled if review_labels[row["pair_id"]] == "not_same"]
    print(f"Reviewed {len(labeled)} pairs; {len(same_pairs)} marked same-property and {len(not_same_pairs)} marked different-property.")

    zeroing_tests = [
        ("semantic median", "semantic_zeroed"),
        ("fuzzy median", "fuzzy_zeroed"),
    ]
    print("For each signal, these counts show how many reviewed examples the median correction sets to zero:")
    for name, zeroed_column in zeroing_tests:
        same_zeroed = sum(row[zeroed_column] for row in same_pairs)
        not_same_zeroed = sum(row[zeroed_column] for row in not_same_pairs)
        print(
            f"{name}: same-property {same_zeroed}/{len(same_pairs)}; "
            f"different-property {not_same_zeroed}/{len(not_same_pairs)}"
        )

    print("Final similarity scores with and without median correction:")
    pprint([
        {
            "pair_id": row["pair_id"],
            "label": review_labels[row["pair_id"]],
            "titles": f'{row["title_a"]} <> {row["title_b"]}',
            "current": round(row["final_current"], 3),
            "no_median": round(row["final_no_median"], 3),
        }
        for row in labeled
    ], width=160)

    print("Raw semantic and fuzzy scores for reviewed pairs; solid lines mark their medians:")
    fig = go.Figure()
    for human_label, color in [("not_same", "#7f8c8d"), ("same", "#d62728")]:
        group = [row for row in labeled if review_labels[row["pair_id"]] == human_label]
        if not group:
            continue
        fig.add_trace(go.Scatter(
            x=[row["semantic_raw"] for row in group],
            y=[row["fuzzy_raw"] for row in group],
            mode="markers+text",
            text=[str(row["pair_id"]) for row in group],
            textposition="top center",
            marker={"size": 10, "color": color},
            name=human_label.replace("_", " "),
            customdata=[[row["title_a"], row["title_b"]] for row in group],
            hovertemplate=(
                "Pair %{text}: %{customdata[0]} <> %{customdata[1]}"
                "<br>Raw semantic: %{x:.3f}<br>Raw fuzzy: %{y:.3f}<extra></extra>"
            ),
        ))
    fig.add_vline(x=semantic_baseline, line_dash="solid", annotation_text="semantic median")
    fig.add_hline(y=fuzzy_baseline, line_dash="solid", annotation_text="fuzzy median")
    fig.update_layout(
        title="Reviewed pairs: raw semantic and fuzzy scores",
        xaxis_title="Raw semantic cosine similarity",
        yaxis_title="Raw fuzzy title similarity",
        xaxis={"range": [-1, 1]},
        yaxis={"range": [0, 1]},
        width=900,
        height=600,
    )
    display(fig)

Reviewed 24 pairs; 3 marked same-property and 21 marked different-property.
For each signal, these counts show how many reviewed examples the median correction sets to zero:
semantic median: same-property 0/3; different-property 12/21
fuzzy median: same-property 3/3; different-property 9/21
Final similarity scores with and without median correction:
[{'current': 0.052, 'label': 'not_same', 'no_median': 0.489, 'pair_id': 0, 'titles': 'High stretch and anatomically optimised fit <> Ultra-lightweight design'},
 {'current': 0.054, 'label': 'not_same', 'no_median': 0.488, 'pair_id': 1, 'titles': 'Premium high-stretch material back <> Invisible under a loose short'},
 {'current': 0.271, 'label': 'not_same', 'no_median': 0.538, 'pair_id': 2, 'titles': 'Silicone-enhanced palm sections <> Comfortable padded palms'},
 {'current': 0.061, 'label': 'not_same', 'no_median': 0.491, 'pair_id': 3, 'titles': 'Comfortable padded palms <> Insulating & Breathable'},
 {'current': 0.083, 'label': 'not_same',

## What the completed review showed

All 24 sampled pairs were reviewed; 3 were marked as same-property: IDs 6, 9, and 11. The median correction kept the semantic signal for all three (0/3 zeroed), but zeroed the fuzzy signal for all three (3/3 zeroed). Their raw semantic scores are 0.793, 0.648, and 0.631, all above the semantic median of about 0.566. Their raw fuzzy scores are 0.298, 0.294, and 0.250, all below the fuzzy median of about 0.318.

This shows the two signals treating these reviewed examples differently. The semantic score uses both titles and their aligned descriptions, so it can retain evidence of a shared property when the titles use different wording. Fuzzy matching compares title strings only, so it gives less credit to those differently worded labels; because each raw fuzzy score is below its median, the correction clips each to zero. This is an observation about these three labeled pairs, not proof that semantic similarity is always more reliable.

The no-median control preserves below-median differences, but its final scores can be higher simply because typical semantic cosine values map to a positive baseline. A higher score by itself does not show improvement; compare how the two versions order the pairs you marked `same` and `not_same`.

## Conclusion and limits

In this small review, median correction did not remove semantic evidence from any of the three same-property pairs, but it did remove fuzzy evidence from all three. This suggests that median calibration affects the fuzzy signal more strongly for these examples, while semantic embeddings retain relevant contextual similarity. The notebook tests only the median correction against a no-median control; it does not test different weights or fixed cutoffs.

The sample was deliberately balanced and contains only three `same` judgments, so these results are exploratory rather than representative evidence for changing the production matrix. The comparison keeps the original weights and other pipeline settings unchanged, and no clustering is run.